# Comparación de cobertura – M1 vs M2
Genera dos parquets de cargas con asignación de hubs (uno por método) y calcula cobertura.

### Inputs esperados
| Variable | Ruta |
|---|---|
| Cargas | `../2. cleaning_loads/parquetloadsfull.parquet` |
| Coords M1 (4 CSVs) | carpeta definida en `M1_CSV_DIR` |
| Coords M2 | `coords_assign_m2.parquet` |

### Outputs
- `loads_assign_m1.parquet` — cargas + asignación M1 origen + destino
- `loads_assign_m2.parquet` — cargas + asignación M2 origen + destino


### Metadata del análisis
| Campo | Valor |
|---|---|
| N registros | 57,573,489 |
| Ventana de fechas | Feb–Ago 2025 |
| Tipo de validación | Within-sample |
| Hubs M1 | 194 (K-Medoids, radio 50 mi) |
| Hubs M2 | 192 (KDE+DBSCAN, radio 50 mi) |

In [1]:
import polars as pl
import pandas as pd
import ast
from pathlib import Path

LOADS_PATH = '../2. cleaning_loads/parquetloadsfull.parquet'
M1_CSV_DIR = '../otros'              # carpeta con los 4 CSVs de M1
M2_COORDS  = 'coords_assign_m2.parquet'
OUT_M1     = 'loads_assign_m1.parquet'
OUT_M2     = 'loads_assign_m2.parquet'


## 1. Coordenadas M1

In [2]:
LIST_COLS = ['hubs_50m', 'hubs_100m', 'hubs_150m']

def load_m1_csv(path):
    df = pd.read_csv(path, dtype={'lat': float, 'lng': float})
    for col in LIST_COLS:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ast.literal_eval(x)
                if pd.notna(x) and str(x).strip() not in ('', '[]') else []
            )
    return df

csvs = sorted(Path(M1_CSV_DIR).glob('*.csv'))
assert csvs, f'No se encontraron CSVs en {M1_CSV_DIR}'

coords_m1 = (
    pd.concat([load_m1_csv(p) for p in csvs], ignore_index=True)
    .drop_duplicates(subset=['lat', 'lng'])
    .reset_index(drop=True)
)
print(f'Coords M1: {len(coords_m1):,} filas  |  archivos: {len(csvs)}')
coords_m1.head()


Coords M1: 163,727 filas  |  archivos: 4


,lat,lng,city,state,ID,near_hub,hubs_50m,hubs_100m,hubs_150m,Unnamed: 0
0,33.893398,-117.28230,MARCH AIR FORCE BASE,CA,493002876,San Diego (CA),[131],"[127, 131]","[125, 127, 131]",NaN
1,33.300970,-87.04774,MC CALLA,AL,494468867,Birmingham (AL),"[1, 6]","[1, 2, 4, 6]","[1, 2, 3, 4, 5, 6, 19, 35, 39]",NaN
2,46.505590,-116.55082,LENORE,ID,494976632,Winchester (ID),[102],"[100, 102]","[100, 102, 134, 153, 158]",NaN
3,45.273250,-122.23474,CURRINSVILLE,OR,492596406,Hood River (OR),"[154, 160]","[152, 154, 160]","[152, 154, 156, 159, 160]",NaN
4,32.886100,-96.43884,ROCKWELL,TX,489224233,Dallas (TX),[185],"[107, 185, 188, 193]","[107, 185, 188, 193]",NaN


## 2. Coordenadas M2

In [3]:
coords_m2 = pl.read_parquet(M2_COORDS)
print(f'Coords M2: {len(coords_m2):,} filas')
print(coords_m2.schema)
coords_m2.head()


Coords M2: 738,827 filas
Schema([('lat', Float64), ('lng', Float64), ('near_hub_m2', String), ('hubs_50m_m2', List(String)), ('hubs_100m_m2', List(String)), ('hubs_150m_m2', List(String))])


lat,lng,near_hub_m2,hubs_50m_m2,hubs_100m_m2,hubs_150m_m2
f64,f64,str,list[str],list[str],list[str]
37.30876,-79.901985,"""71.0""","[""71.0""]","[""71.0"", ""201.0"", ""C181""]","[""93.0"", ""186.0"", … ""C181""]"
35.186402,-84.828057,"""65.0""","[""65.0""]","[""36.0"", ""63.0"", ""65.0""]","[""1.0"", ""36.0"", … ""C110""]"
34.4734,-82.656997,"""158.0""",[],"[""158.0"", ""66.0"", ""C110""]","[""36.0"", ""93.0"", … ""C110""]"
41.712752,-93.591708,"""38.0""","[""38.0""]","[""38.0"", ""72.0""]","[""38.0"", ""72.0"", … ""C126""]"
39.76899,-104.76573,"""16.0""","[""16.0""]","[""16.0"", ""C102""]","[""16.0"", ""221.0"", … ""C149""]"


## 3. Parquet de cargas con asignación M1
Merge doble: primero por coordenadas de origen, luego por coordenadas de destino.

In [4]:
def m1_to_pl(df, suffix):
    rename = {
        'near_hub':  f'near_hub_m1_{suffix}',
        'hubs_50m':  f'hubs_50m_m1_{suffix}',
        'hubs_100m': f'hubs_100m_m1_{suffix}',
        'hubs_150m': f'hubs_150m_m1_{suffix}',
    }
    keep = ['lat', 'lng'] + [c for c in rename if c in df.columns]
    return (
        pl.from_pandas(df[keep])
        .rename({c: rename[c] for c in rename if c in df.columns})
        .with_columns([
            pl.col('lat').cast(pl.Float64),
            pl.col('lng').cast(pl.Float64),
        ])
    )

m1_orig = m1_to_pl(coords_m1, 'orig')
m1_dest = m1_to_pl(coords_m1, 'dest')

loads_m1 = (
    pl.scan_parquet(LOADS_PATH)
    .with_columns([
        pl.col('LatOrigin').cast(pl.Float64),
        pl.col('LngOrigin').cast(pl.Float64),
        pl.col('LatDestination').cast(pl.Float64),
        pl.col('LngDestination').cast(pl.Float64),
    ])
    .join(m1_orig.lazy(), left_on=['LatOrigin', 'LngOrigin'],
          right_on=['lat', 'lng'], how='left')
    .join(m1_dest.lazy(), left_on=['LatDestination', 'LngDestination'],
          right_on=['lat', 'lng'], how='left')
    .collect()
)

loads_m1.write_parquet(OUT_M1)
print(f'Guardado: {OUT_M1}  ({len(loads_m1):,} filas)')
print(loads_m1.select([c for c in loads_m1.columns if 'm1' in c]).head())


Guardado: loads_assign_m1.parquet  (57,573,489 filas)
shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ near_hub_m ┆ hubs_50m_m ┆ hubs_100m_ ┆ hubs_150m ┆ near_hub_ ┆ hubs_50m_ ┆ hubs_100m ┆ hubs_150m │
│ 1_orig     ┆ 1_orig     ┆ m1_orig    ┆ _m1_orig  ┆ m1_dest   ┆ m1_dest   ┆ _m1_dest  ┆ _m1_dest  │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ list[str]  ┆ list[str]  ┆ list[str] ┆ str       ┆ list[str] ┆ list[str] ┆ list[str] │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Birmingham ┆ ["6"]      ┆ ["1", "2", ┆ ["1",     ┆ Athens    ┆ ["19",    ┆ ["19",    ┆ ["2",     │
│ (AL)       ┆            ┆ … "6"]     ┆ "2", …    ┆ (GA)      ┆ "21"]     ┆ "21", …   ┆ "6", …    │
│            ┆            ┆            ┆ "142"]    ┆           ┆           ┆ "23"]     ┆ "141"]    │
│ Milwaukee  ┆ ["118"] 

## 4. Parquet de cargas con asignación M2

In [5]:
def m2_to_pl(df, suffix):
    rename = {
        'near_hub_m2':  f'near_hub_m2_{suffix}',
        'hubs_50m_m2':  f'hubs_50m_m2_{suffix}',
        'hubs_100m_m2': f'hubs_100m_m2_{suffix}',
        'hubs_150m_m2': f'hubs_150m_m2_{suffix}',
    }
    keep = ['lat', 'lng'] + [c for c in rename if c in df.columns]
    return (
        df.select(keep)
        .rename({c: rename[c] for c in rename if c in df.columns})
        .with_columns([
            pl.col('lat').cast(pl.Float64),
            pl.col('lng').cast(pl.Float64),
        ])
    )

m2_orig = m2_to_pl(coords_m2, 'orig')
m2_dest = m2_to_pl(coords_m2, 'dest')

loads_m2 = (
    pl.scan_parquet(LOADS_PATH)
    .with_columns([
        pl.col('LatOrigin').cast(pl.Float64),
        pl.col('LngOrigin').cast(pl.Float64),
        pl.col('LatDestination').cast(pl.Float64),
        pl.col('LngDestination').cast(pl.Float64),
    ])
    .join(m2_orig.lazy(), left_on=['LatOrigin', 'LngOrigin'],
          right_on=['lat', 'lng'], how='left')
    .join(m2_dest.lazy(), left_on=['LatDestination', 'LngDestination'],
          right_on=['lat', 'lng'], how='left')
    .collect()
)

loads_m2.write_parquet(OUT_M2)
print(f'Guardado: {OUT_M2}  ({len(loads_m2):,} filas)')
print(loads_m2.select([c for c in loads_m2.columns if 'm2' in c]).head())


Guardado: loads_assign_m2.parquet  (57,573,489 filas)
shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ near_hub_m ┆ hubs_50m_m ┆ hubs_100m_ ┆ hubs_150m ┆ near_hub_ ┆ hubs_50m_ ┆ hubs_100m ┆ hubs_150m │
│ 2_orig     ┆ 2_orig     ┆ m2_orig    ┆ _m2_orig  ┆ m2_dest   ┆ m2_dest   ┆ _m2_dest  ┆ _m2_dest  │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ list[str]  ┆ list[str]  ┆ list[str] ┆ str       ┆ list[str] ┆ list[str] ┆ list[str] │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ C174       ┆ ["C174"]   ┆ ["C153",   ┆ ["C153",  ┆ 11.0      ┆ ["11.0"]  ┆ ["11.0",  ┆ ["11.0",  │
│            ┆            ┆ "C171",    ┆ "C170", … ┆           ┆           ┆ "C96", …  ┆ "C96", …  │
│            ┆            ┆ "C174"]    ┆ "C174"]   ┆           ┆           ┆ "C100"]   ┆ "C100"]   │
│ C179       ┆ ["C179"]

## 5. Cobertura a 50 millas

In [6]:
import gc

def coverage(parquet_path, col_50m, col_lat):
    # scan_parquet: solo trae las 2 columnas necesarias, no el parquet completo
    df = (
        pl.scan_parquet(parquet_path)
        .select([col_lat, col_50m])
        .collect()
    )
    total      = len(df)
    con_coords = df.filter(pl.col(col_lat).is_not_null()).height
    cubiertos  = df.filter(pl.col(col_50m).list.len() > 0).height
    del df; gc.collect()
    return {
        'total_cargas':   total,
        'con_coordenada': con_coords,
        'cubiertos_50m':  cubiertos,
        'pct_total':      round(100 * cubiertos / total, 1),
        'pct_con_coord':  round(100 * cubiertos / con_coords, 1) if con_coords else 0,
    }

stats = {
    'M1 origen':  coverage(OUT_M1, 'hubs_50m_m1_orig', 'LatOrigin'),
    'M1 destino': coverage(OUT_M1, 'hubs_50m_m1_dest', 'LatDestination'),
    'M2 origen':  coverage(OUT_M2, 'hubs_50m_m2_orig', 'LatOrigin'),
    'M2 destino': coverage(OUT_M2, 'hubs_50m_m2_dest', 'LatDestination'),
}

print(pd.DataFrame(stats).T.to_string())


            total_cargas  con_coordenada  cubiertos_50m  pct_total  pct_con_coord
M1 origen     57573489.0      57046115.0     48216538.0       83.7           84.5
M1 destino    57573489.0      57038026.0     48031623.0       83.4           84.2
M2 origen     57573489.0      57046115.0     46740304.0       81.2           81.9
M2 destino    57573489.0      57038026.0     47801351.0       83.0           83.8


## 6. Solapamiento — ¿cuántos hubs cubre cada carga cubierta?\n\nSi una carga tiene 3 hubs dentro de 50mi, está triplicada en la red.\nUn solapamiento alto significa redundancia: quitar un hub no descubre apenas nada.\n\n- **1 hub**: cobertura exclusiva — ese hub es irremplazable para esa carga\n- **2+ hubs**: solapamiento — esa carga la cubren varios hubs a la vez

In [7]:
def overlap_stats(parquet_path, col_50m):
    """Distribución de solapamiento + suma de eventos en un solo scan."""
    df = (
        pl.scan_parquet(parquet_path)
        .select(pl.col(col_50m).list.len().alias('n_hubs'))
        .filter(pl.col('n_hubs') > 0)
        .collect()
    )
    total    = len(df)
    eventos  = int(df['n_hubs'].sum())
    df = df.with_columns(
        pl.when(pl.col('n_hubs') >= 4).then(4).otherwise(pl.col('n_hubs')).alias('bin')
    )
    dist = (
        df.group_by('bin').agg(pl.len().alias('cargas')).sort('bin')
        .with_columns((pl.col('cargas') / total * 100).round(1).alias('%'))
    )
    del df; gc.collect()
    return dist, total, eventos

dist_m1, tot_m1, ev_m1 = overlap_stats(OUT_M1, 'hubs_50m_m1_orig')
dist_m2, tot_m2, ev_m2 = overlap_stats(OUT_M2, 'hubs_50m_m2_orig')

print(f'{"n_hubs":>8}  {"M1 cargas":>12}  {"M1 %":>7}  {"M2 cargas":>12}  {"M2 %":>7}')
print('-' * 57)
labels = {1: '1 (excl)', 2: '2', 3: '3', 4: '4+'}
m1_d = {r['bin']: r for r in dist_m1.to_dicts()}
m2_d = {r['bin']: r for r in dist_m2.to_dicts()}
for k, label in labels.items():
    r1 = m1_d.get(k, {'cargas': 0, '%': 0.0})
    r2 = m2_d.get(k, {'cargas': 0, '%': 0.0})
    print(f'{label:>8}  {r1["cargas"]:>12,}  {r1["%"]:>6.1f}%  {r2["cargas"]:>12,}  {r2["%"]:>6.1f}%')
print('-' * 57)
print(f'{"Total":>8}  {tot_m1:>12,}  {"100.0%":>7}  {tot_m2:>12,}  {"100.0%":>7}')

print(f'\nEficiencia de red (cargas únicas / eventos de cobertura):')
print(f'  M1: {tot_m1:,} / {ev_m1:,} = {tot_m1/ev_m1:.3f}')
print(f'  M2: {tot_m2:,} / {ev_m2:,} = {tot_m2/ev_m2:.3f}')
print('  (1.0 = sin solapamiento; < 1.0 = redundancia)')


  n_hubs     M1 cargas     M1 %     M2 cargas     M2 %
---------------------------------------------------------
1 (excl)    42,473,965    88.1%    43,049,230    92.1%
       2     5,741,327    11.9%     3,691,074     7.9%
       3         1,246     0.0%             0     0.0%
      4+             0     0.0%             0     0.0%
---------------------------------------------------------
   Total    48,216,538   100.0%    46,740,304   100.0%

Eficiencia de red (cargas únicas / eventos de cobertura):
  M1: 48,216,538 / 53,960,357 = 0.894
  M2: 46,740,304 / 50,431,378 = 0.927
  (1.0 = sin solapamiento; < 1.0 = redundancia)


## 7. Relevancia por hub (M2)\n\nPara cada hub M2: total de cargas que cubre, cuántas de forma exclusiva (solo él las cubre),\ny el % de exclusividad. Un hub con exclusividad baja es redundante — si lo quitamos, casi\ntodas sus cargas siguen siendo cubiertas por otro hub.

In [8]:
# Cargar solo columnas necesarias para M2 origen
m2_hubs = (
    pl.scan_parquet(OUT_M2)
    .select(['near_hub_m2_orig', 'hubs_50m_m2_orig'])
    .filter(pl.col('hubs_50m_m2_orig').list.len() > 0)
    .collect()
)

# Total de cargas por hub (near_hub = el más cercano, proxy del "hub responsable")
total_per_hub = (
    m2_hubs
    .group_by('near_hub_m2_orig')
    .agg(pl.len().alias('total'))
)

# Cargas exclusivas: list.len() == 1 → solo un hub las cubre
exclusive_per_hub = (
    m2_hubs
    .filter(pl.col('hubs_50m_m2_orig').list.len().eq(1))
    .group_by('near_hub_m2_orig')
    .agg(pl.len().alias('exclusive'))
)

hub_stats = (
    total_per_hub
    .join(exclusive_per_hub, on='near_hub_m2_orig', how='left')
    .with_columns(pl.col('exclusive').fill_null(0))
    .with_columns(
        (pl.col('exclusive') / pl.col('total') * 100).round(1).alias('pct_exclusive')
    )
    .sort('total', descending=True)
    .rename({'near_hub_m2_orig': 'hub_id'})
)

print(f'Hubs M2 con cargas asignadas: {len(hub_stats)}')
print(f'\nTop 20 por volumen:')
print(hub_stats.head(20).to_pandas().to_string(index=False))

print(f'\nResumen de exclusividad:')
print(f'  Hubs con >80% exclusiva:  {(hub_stats["pct_exclusive"] > 80).sum()}')
print(f'  Hubs con 50-80% exclusiva: {((hub_stats["pct_exclusive"] >= 50) & (hub_stats["pct_exclusive"] <= 80)).sum()}')
print(f'  Hubs con <50% exclusiva:  {(hub_stats["pct_exclusive"] < 50).sum()}')
print(f'  Hubs con 0% exclusiva:    {(hub_stats["pct_exclusive"] == 0).sum()}  ← completamente redundantes')

Hubs M2 con cargas asignadas: 184

Top 20 por volumen:
hub_id   total  exclusive  pct_exclusive
  11.0 1727222    1613591           93.4
 183.0 1719795    1719773          100.0
  42.0 1677184    1531393           91.3
  58.0 1558355    1558355          100.0
 119.0 1183852    1173440           99.1
  36.0 1146424    1038052           90.5
  86.0  816856     751449           92.0
  66.0  803869     800500           99.6
 137.0  799182     570880           71.4
  18.0  736315     504048           68.5
  87.0  728329     564655           77.5
 180.0  723359     723068          100.0
  82.0  692552     674822           97.4
  93.0  653351     521658           79.8
  74.0  651465     651465          100.0
   1.0  589750     589692          100.0
  47.0  587740     584605           99.5
 193.0  579543     527788           91.1
 128.0  578114     530482           91.8
 131.0  572942     553437           96.6

Resumen de exclusividad:
  Hubs con >80% exclusiva:  159
  Hubs con 50-80% exclusiv

In [9]:
# Concentración: top-N hubs cubren qué % del total de cargas cubiertas
total_cubiertas_m2 = hub_stats['total'].sum()

print('Concentración de carga por top-N hubs M2:')
for n in [10, 20, 30, 50]:
    pct = hub_stats.head(n)['total'].sum() / total_cubiertas_m2 * 100
    print(f'  Top {n:>3} hubs → {pct:.1f}% de las cargas cubiertas')

# Mismo análisis para M1
m1_hubs = (
    pl.scan_parquet(OUT_M1)
    .select(['near_hub_m1_orig', 'hubs_50m_m1_orig'])
    .filter(pl.col('hubs_50m_m1_orig').list.len() > 0)
    .collect()
)
m1_hub_stats = (
    m1_hubs
    .group_by('near_hub_m1_orig')
    .agg(pl.len().alias('total'))
    .sort('total', descending=True)
)
total_cubiertas_m1 = m1_hub_stats['total'].sum()

print('\nConcentración de carga por top-N hubs M1:')
for n in [10, 20, 30, 50]:
    pct = m1_hub_stats.head(n)['total'].sum() / total_cubiertas_m1 * 100
    print(f'  Top {n:>3} hubs → {pct:.1f}% de las cargas cubiertas')

del m2_hubs, m1_hubs

Concentración de carga por top-N hubs M2:
  Top  10 hubs → 26.0% de las cargas cubiertas
  Top  20 hubs → 39.6% de las cargas cubiertas
  Top  30 hubs → 50.6% de las cargas cubiertas
  Top  50 hubs → 66.7% de las cargas cubiertas

Concentración de carga por top-N hubs M1:
  Top  10 hubs → 23.5% de las cargas cubiertas
  Top  20 hubs → 35.5% de las cargas cubiertas
  Top  30 hubs → 45.0% de las cargas cubiertas
  Top  50 hubs → 60.3% de las cargas cubiertas


In [10]:
# Exclusividad por hub M1 — mismo análisis que M2
m1_hubs_excl = (
    pl.scan_parquet(OUT_M1)
    .select(['near_hub_m1_orig', 'hubs_50m_m1_orig'])
    .filter(pl.col('hubs_50m_m1_orig').list.len() > 0)
    .collect()
)

total_per_hub_m1 = (
    m1_hubs_excl
    .group_by('near_hub_m1_orig')
    .agg(pl.len().alias('total'))
)
exclusive_per_hub_m1 = (
    m1_hubs_excl
    .filter(pl.col('hubs_50m_m1_orig').list.len().eq(1))
    .group_by('near_hub_m1_orig')
    .agg(pl.len().alias('exclusive'))
)
hub_stats_m1 = (
    total_per_hub_m1
    .join(exclusive_per_hub_m1, on='near_hub_m1_orig', how='left')
    .with_columns(pl.col('exclusive').fill_null(0))
    .with_columns(
        (pl.col('exclusive') / pl.col('total') * 100).round(1).alias('pct_exclusive')
    )
    .sort('total', descending=True)
    .rename({'near_hub_m1_orig': 'hub_id'})
)

print(f'Hubs M1 con cargas asignadas: {len(hub_stats_m1)}')
print(f'\nResumen de exclusividad M1:')
print(f'  Hubs con >80% exclusiva:   {(hub_stats_m1["pct_exclusive"] > 80).sum()}')
print(f'  Hubs con 50-80% exclusiva: {((hub_stats_m1["pct_exclusive"] >= 50) & (hub_stats_m1["pct_exclusive"] <= 80)).sum()}')
print(f'  Hubs con <50% exclusiva:   {(hub_stats_m1["pct_exclusive"] < 50).sum()}')
print(f'  Hubs con 0% exclusiva:     {(hub_stats_m1["pct_exclusive"] == 0).sum()}  ← completamente redundantes')

print(f'\nComparación M1 vs M2:')
print(f'  {"Categoría":<25} {"M1":>6}  {"M2":>6}')
print(f'  {"-"*38}')
for lbl, lo, hi in [('> 80% excl.', 80, 101), ('50–80% excl.', 50, 80), ('< 50% excl.', 0, 50)]:
    n1 = ((hub_stats_m1['pct_exclusive'] >= lo) & (hub_stats_m1['pct_exclusive'] < hi)).sum()
    n2 = ((hub_stats['pct_exclusive']    >= lo) & (hub_stats['pct_exclusive']    < hi)).sum()
    print(f'  {lbl:<25} {n1:>6}  {n2:>6}')

del m1_hubs_excl; gc.collect()

Hubs M1 con cargas asignadas: 194

Resumen de exclusividad M1:
  Hubs con >80% exclusiva:   163
  Hubs con 50-80% exclusiva: 20
  Hubs con <50% exclusiva:   11
  Hubs con 0% exclusiva:     0  ← completamente redundantes

Comparación M1 vs M2:
  Categoría                     M1      M2
  --------------------------------------
  > 80% excl.                  164     159
  50–80% excl.                  19      21
  < 50% excl.                   11       4


0

## 8. Similitud de centroides M1 vs M2

Comparación geométrica: para cada hub de un método, ¿cuál es el hub más cercano del otro método y a qué distancia?

Una mediana baja (< 20 mi) significaría que ambos métodos eligieron ubicaciones casi idénticas. Una mediana alta (> 50 mi) indica estrategias de posicionamiento distintas.

In [ ]:
import os
from sklearn.neighbors import BallTree

# ── Centroides M1 (K-Medoids, hubs oficiales) ───────────────────────────────
hubs_m1 = pd.read_excel('../utils/hubs_id.xlsx', sheet_name=0, engine='openpyxl')
# Columnas: id, old_name, lat, lng, new_name, state, state_name, region
print(f'Hubs M1: {len(hubs_m1)}')

# ── Centroides M2 (KDE+DBSCAN, optimizados, sin imputados) ──────────────────
m2_path = 'final_results_optimized.csv' if os.path.exists('final_results_optimized.csv') else 'final_results.csv'
hubs_m2 = pd.read_csv(m2_path).dropna(subset=['lat', 'lng'])
print(f'Hubs M2: {len(hubs_m2)}  (desde {m2_path})')

# ── Similitud nearest-neighbor (haversine) ───────────────────────────────────
m1_coords_rad = np.column_stack([np.deg2rad(hubs_m1['lat'].values), np.deg2rad(hubs_m1['lng'].values)])
m2_coords_rad = np.column_stack([np.deg2rad(hubs_m2['lat'].values), np.deg2rad(hubs_m2['lng'].values)])

tree_m1 = BallTree(m1_coords_rad, metric='haversine')
tree_m2 = BallTree(m2_coords_rad, metric='haversine')

# Para cada M2 hub → hub M1 más cercano
d_m2_to_m1_rad, _ = tree_m1.query(m2_coords_rad, k=1)
d_m2_to_m1 = d_m2_to_m1_rad.ravel() * 3958.8

# Para cada M1 hub → hub M2 más cercano
d_m1_to_m2_rad, _ = tree_m2.query(m1_coords_rad, k=1)
d_m1_to_m2 = d_m1_to_m2_rad.ravel() * 3958.8

print('\n── Distancia M2→M1 (cada hub M2 a su hub M1 más cercano) ──')
print(f'  Media:   {d_m2_to_m1.mean():.1f} mi')
print(f'  Mediana: {np.median(d_m2_to_m1):.1f} mi')
print(f'  P75:     {np.percentile(d_m2_to_m1, 75):.1f} mi')
print(f'  Máx:     {d_m2_to_m1.max():.1f} mi')
print(f'  % dentro de 50 mi: {(d_m2_to_m1 <= 50).mean()*100:.1f}%')

print('\n── Distancia M1→M2 (cada hub M1 a su hub M2 más cercano) ──')
print(f'  Media:   {d_m1_to_m2.mean():.1f} mi')
print(f'  Mediana: {np.median(d_m1_to_m2):.1f} mi')
print(f'  P75:     {np.percentile(d_m1_to_m2, 75):.1f} mi')
print(f'  Máx:     {d_m1_to_m2.max():.1f} mi')
print(f'  % dentro de 50 mi: {(d_m1_to_m2 <= 50).mean()*100:.1f}%')


## 9. Mapa Folium — centroides M1 vs M2 (capas independientes)

In [ ]:
import folium
from IPython.display import display

m = folium.Map(location=[38, -96], zoom_start=4, tiles='OpenStreetMap')

fg_m1 = folium.FeatureGroup(name='V1 — K-Medoids (194 hubs)', show=True)
fg_m2 = folium.FeatureGroup(name='V2 — KDE+DBSCAN', show=True)

for _, row in hubs_m1.iterrows():
    name = row.get('new_name') or row.get('old_name') or str(row['id'])
    folium.CircleMarker(
        [row['lat'], row['lng']],
        radius=5, color='steelblue', fill=True, fill_opacity=0.7,
        tooltip=f"V1 ({row.get('state', '')}): {name}"
    ).add_to(fg_m1)

for _, row in hubs_m2.iterrows():
    name = row.get('new_name') or str(row.get('id', ''))
    folium.CircleMarker(
        [row['lat'], row['lng']],
        radius=5, color='darkorange', fill=True, fill_opacity=0.7,
        tooltip=f"V2 ({row.get('state', '')}): {name}"
    ).add_to(fg_m2)

fg_m1.add_to(m)
fg_m2.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
display(m)
